In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import os
import subprocess
import threading
import pygame
import wave
import shutil

# --- 核心參數設定 ---
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

class MusicFilterTest:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂過濾測試版")
        self.root.geometry("600x550")
        
        pygame.mixer.init()
        
        # 變數區
        self.input_wav = tk.StringVar(value="")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        self.current_pos = 0.0  # 當前播放秒數
        self.is_playing = False
        self.total_duration = 0.0
        
        # 音軌控制變數 (Bass, Drums, Other, Vocals)
        self.tracks = {
            "bass": {"var": tk.BooleanVar(value=True), "file": ""},
            "drums": {"var": tk.BooleanVar(value=True), "file": ""},
            "other": {"var": tk.BooleanVar(value=True), "file": ""},
            "vocals": {"var": tk.BooleanVar(value=True), "file": ""}
        }
        
        self.setup_ui()

    def setup_ui(self):
        # 1. 檔案選擇
        frame_file = ttk.LabelFrame(self.root, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        # 2. AI 分離控制
        frame_ai = ttk.LabelFrame(self.root, text=" 2. AI 樂器分離 ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        
        self.btn_start_ai = ttk.Button(frame_ai, text="啟動分離 (Demucs)", command=self.start_separation_thread)
        self.btn_start_ai.pack(pady=5)
        
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        # 3. 播放控制與進度條
        frame_play = ttk.LabelFrame(self.root, text=" 3. 同步撥放控制 ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=5)
        self.btn_play = ttk.Button(btn_box, text="播放 / 暫停", command=self.toggle_play)
        self.btn_play.grid(row=0, column=1, padx=5)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=2, padx=5)

        # 4. 音軌開關 (Radio/Checkbox 概念)
        frame_tracks = ttk.LabelFrame(self.root, text=" 4. 樂器選擇 (更改將回歸初始時間) ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        
        for name, data in self.tracks.items():
            cb = ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                                 command=self.on_track_change)
            cb.pack(side="left", expand=True)

    # --- 邏輯功能 ---

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav")])
        if path:
            self.input_wav.set(path)
            # 獲取總時長
            with wave.open(path, 'r') as f:
                self.total_duration = f.getnframes() / float(f.getframerate())
            self.update_time_label()

    def start_separation_thread(self):
        if not self.input_wav.get():
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: return
        
        # 清除舊音軌
        self.stop_playback()
        self.progress_val.set(0)
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.lbl_status.config(text="AI 正在處理中，請稍候...")
        
        input_path = self.input_wav.get()
        temp_name = "filter_temp.wav"
        shutil.copy(input_path, temp_name)
        
        # 建立專屬輸出資料夾
        job_output = os.path.join(OUTPUT_BASE, "filter_session")
        if os.path.exists(job_output):
            shutil.rmtree(job_output) # 清除上一次的

        cmd = ['demucs', '-d', 'cuda', '-n', 'mdx_extra', temp_name, '-o', job_output]
        
        try:
            # 這裡用一個簡單的虛擬進度跑條，因為 Demucs 的 stdout 較難精確解析 %
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            
            # 模擬進度條前進
            while process.poll() is None:
                curr = self.progress_val.get()
                if curr < 90: self.progress_val.set(curr + 0.5)
                self.root.update_idletasks()
                process.stdout.readline() # 讀取輸出避免阻塞
            
            if process.returncode == 0:
                self.progress_val.set(100)
                # 設定音軌路徑
                track_dir = os.path.join(job_output, "mdx_extra", "filter_temp")
                for name in self.tracks.keys():
                    self.tracks[name]["file"] = os.path.join(track_dir, f"{name}.wav")
                
                self.lbl_status.config(text="分離完成！可以開始播放。")
                messagebox.showinfo("成功", "樂器分離已完成")
            else:
                self.lbl_status.config(text="分離失敗。")
        except Exception as e:
            self.lbl_status.config(text=f"錯誤: {e}")
        finally:
            if os.path.exists(temp_name): os.remove(temp_name)
            self.is_processing = False
            self.btn_start_ai.config(state="normal")

    def toggle_play(self):
        if not self.tracks["vocals"]["file"]:
            messagebox.showwarning("提示", "請先執行 AI 分離")
            return
            
        if self.is_playing:
            self.pause_playback()
        else:
            self.start_playback()

    def start_playback(self):
        pygame.mixer.stop()
        pygame.mixer.set_num_channels(4)
        
        # 同步啟動所有被勾選的音軌
        for i, (name, data) in enumerate(self.tracks.items()):
            if data["var"].get():
                sound = pygame.mixer.Sound(data["file"])
                chan = pygame.mixer.Channel(i)
                chan.play(sound)
                chan.set_pos(self.current_pos) # 跳轉到當前位置
            
        self.is_playing = True
        self.update_sync_ui()

    def pause_playback(self):
        # 獲取當前播放位置 (Pygame mixer 不好拿精確位置，所以我們用時間計算)
        pygame.mixer.stop()
        self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_time_label()

    def on_track_change(self):
        """當使用者增減音軌時，依照你的要求回歸初始時間 (0)"""
        was_playing = self.is_playing
        self.stop_playback()
        self.current_pos = 0 # 回歸初始
        self.play_pb['value'] = 0
        self.update_time_label()
        if was_playing:
            self.start_playback()

    def seek(self, seconds):
        self.current_pos += seconds
        if self.current_pos < 0: self.current_pos = 0
        if self.current_pos > self.total_duration: self.current_pos = self.total_duration
        
        if self.is_playing:
            self.start_playback()
        else:
            self.update_time_label()

    def update_sync_ui(self):
        if self.is_playing:
            self.current_pos += 0.1
            if self.current_pos >= self.total_duration:
                self.stop_playback()
            else:
                self.update_time_label()
                self.root.after(100, self.update_sync_ui)

    def update_time_label(self):
        cur_m, cur_s = divmod(int(self.current_pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0:
            self.play_pb['value'] = (self.current_pos / self.total_duration) * 100

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicFilterTest(root)
    root.mainloop()

C:\Users\jt725\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, scrolledtext
import os
import subprocess
import threading
import pygame
import wave
import shutil
import time
import uuid
import numpy as np
from scipy.io import wavfile

# --- 核心參數設定 ---
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

class AudioSegment:
    """簡易音訊片段類別,支援從指定位置開始播放"""
    def __init__(self, filepath):
        self.filepath = filepath
        self.rate, self.data = wavfile.read(filepath)
        # 轉換為 float32 並正規化
        if self.data.dtype == np.int16:
            self.data = self.data.astype(np.float32) / 32768.0
        elif self.data.dtype == np.int32:
            self.data = self.data.astype(np.float32) / 2147483648.0
    
    def get_segment(self, start_seconds):
        """取得從 start_seconds 開始的音訊片段"""
        start_frame = int(start_seconds * self.rate)
        if start_frame >= len(self.data):
            # 如果起始點超過長度,返回靜音
            return np.zeros((1000, self.data.shape[1] if len(self.data.shape) > 1 else 1), dtype=np.float32)
        
        segment_data = self.data[start_frame:]
        
        # 轉回 int16 給 pygame
        segment_int16 = (segment_data * 32767).astype(np.int16)
        
        # 建立臨時 wav
        temp_file = f"temp_segment_{uuid.uuid4().hex[:8]}.wav"
        wavfile.write(temp_file, self.rate, segment_int16)
        
        return temp_file

class MusicFilterTest:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂過濾器 - 跳轉修復版")
        self.root.geometry("700x700")
        
        # 初始化音訊設定
        pygame.mixer.pre_init(44100, -16, 2, 2048)
        pygame.mixer.init()
        pygame.mixer.set_num_channels(8)
        
        self.input_wav = tk.StringVar(value="")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        
        # 播放控制變數
        self.current_pos = 0.0
        self.start_tick = 0
        self.is_playing = False
        self.total_duration = 0.0
        
        # 音軌路徑與控制
        self.tracks = {
            "bass":   {"var": tk.BooleanVar(value=True), "file": "", "segment": None, "sound": None},
            "drums":  {"var": tk.BooleanVar(value=True), "file": "", "segment": None, "sound": None},
            "other":  {"var": tk.BooleanVar(value=True), "file": "", "segment": None, "sound": None},
            "vocals": {"var": tk.BooleanVar(value=True), "file": "", "segment": None, "sound": None}
        }
        
        self.audio_segments = {}  # 儲存完整音訊資料
        self.temp_files = []  # 追蹤臨時檔案
        self.demucs_process = None
        
        self.setup_ui()

    def setup_ui(self):
        # 1. 檔案選擇
        frame_file = ttk.LabelFrame(self.root, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        # 2. AI 分離控制
        frame_ai = ttk.LabelFrame(self.root, text=" 2. AI 樂器分離 ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        
        # 模型選擇
        model_frame = ttk.Frame(frame_ai)
        model_frame.pack(fill="x", pady=5)
        ttk.Label(model_frame, text="模型:").pack(side="left", padx=5)
        self.model_var = tk.StringVar(value="htdemucs")
        ttk.Radiobutton(model_frame, text="htdemucs (快)", variable=self.model_var, value="htdemucs").pack(side="left", padx=5)
        ttk.Radiobutton(model_frame, text="mdx_extra (優)", variable=self.model_var, value="mdx_extra").pack(side="left", padx=5)
        
        # 裝置選擇
        device_frame = ttk.Frame(frame_ai)
        device_frame.pack(fill="x", pady=5)
        ttk.Label(device_frame, text="裝置:").pack(side="left", padx=5)
        self.device_var = tk.StringVar(value="cpu")
        ttk.Radiobutton(device_frame, text="CPU", variable=self.device_var, value="cpu").pack(side="left", padx=5)
        ttk.Radiobutton(device_frame, text="CUDA", variable=self.device_var, value="cuda").pack(side="left", padx=5)
        
        btn_container = ttk.Frame(frame_ai)
        btn_container.pack(fill="x", pady=5)
        self.btn_start_ai = ttk.Button(btn_container, text="啟動分離", command=self.start_separation_thread)
        self.btn_start_ai.pack(side="left", padx=5)
        self.btn_cancel = ttk.Button(btn_container, text="取消", command=self.cancel_processing, state="disabled")
        self.btn_cancel.pack(side="left", padx=5)
        
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        # 3. 日誌視窗
        frame_log = ttk.LabelFrame(self.root, text=" 處理日誌 ", padding=10)
        frame_log.pack(fill="both", expand=True, padx=10, pady=5)
        
        self.log_text = scrolledtext.ScrolledText(frame_log, height=8, wrap=tk.WORD, font=("Consolas", 9))
        self.log_text.pack(fill="both", expand=True)
        
        ttk.Button(frame_log, text="清除日誌", command=lambda: self.log_text.delete(1.0, tk.END)).pack(pady=5)

        # 4. 播放控制
        frame_play = ttk.LabelFrame(self.root, text=" 3. 同步播放控制 (支援精確跳轉) ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=3)
        ttk.Button(btn_box, text="<< 1s", command=lambda: self.seek(-1)).grid(row=0, column=1, padx=3)
        self.btn_play = ttk.Button(btn_box, text="播放 / 暫停", command=self.toggle_play)
        self.btn_play.grid(row=0, column=2, padx=3)
        ttk.Button(btn_box, text=">> 1s", command=lambda: self.seek(1)).grid(row=0, column=3, padx=3)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=4, padx=3)
        ttk.Button(btn_box, text="停止", command=self.stop_playback).grid(row=0, column=5, padx=3)

        # 5. 音軌開關
        frame_tracks = ttk.LabelFrame(self.root, text=" 4. 樂器選擇 ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        for name, data in self.tracks.items():
            ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                            command=self.on_track_change).pack(side="left", expand=True)

    def log(self, message):
        timestamp = time.strftime("%H:%M:%S")
        self.log_text.insert(tk.END, f"[{timestamp}] {message}\n")
        self.log_text.see(tk.END)
        self.root.update_idletasks()

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav"), ("所有檔案", "*.*")])
        if path:
            self.input_wav.set(path)
            try:
                with wave.open(path, 'r') as f:
                    self.total_duration = f.getnframes() / float(f.getframerate())
                self.log(f"✓ 已載入: {os.path.basename(path)} ({int(self.total_duration//60)}:{int(self.total_duration%60):02d})")
                self.stop_playback()
            except Exception as e:
                self.log(f"✗ 讀取失敗: {e}")
                messagebox.showerror("錯誤", f"無法讀取檔案:\n{e}")

    def start_separation_thread(self):
        if not self.input_wav.get():
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: 
            messagebox.showwarning("提示", "正在處理中")
            return
        
        self.stop_playback()
        self.progress_val.set(0)
        self.log("=" * 50)
        self.log(f"開始分離 (模型: {self.model_var.get()}, 裝置: {self.device_var.get()})")
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def cancel_processing(self):
        if self.demucs_process and self.demucs_process.poll() is None:
            self.log("! 正在取消...")
            try:
                self.demucs_process.terminate()
                time.sleep(1)
                if self.demucs_process.poll() is None:
                    self.demucs_process.kill()
                self.log("✓ 已取消")
            except Exception as e:
                self.log(f"! 取消錯誤: {e}")
            
            self.lbl_status.config(text="已取消")
            self.is_processing = False
            self.btn_start_ai.config(state="normal")
            self.btn_cancel.config(state="disabled")

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.btn_cancel.config(state="normal")
        self.lbl_status.config(text="處理中...")
        
        input_path = self.input_wav.get()
        temp_name = f"filter_temp_{uuid.uuid4().hex[:8]}.wav"
        
        try:
            self.log(f"→ 複製檔案...")
            shutil.copy(input_path, temp_name)
            file_size = os.path.getsize(temp_name) / (1024*1024)
            self.log(f"✓ 暫存: {temp_name} ({file_size:.1f} MB)")
            
            job_output = os.path.join(OUTPUT_BASE, "filter_session")
            if os.path.exists(job_output):
                shutil.rmtree(job_output)
            os.makedirs(job_output, exist_ok=True)
            self.log(f"✓ 輸出: {job_output}")

            model = self.model_var.get()
            device = self.device_var.get()
            cmd = ['demucs', '-d', device, '-n', model, temp_name, '-o', job_output]
            
            self.log(f"→ 執行 demucs...")
            self.log(f"→ 請耐心等候 (可能需要數分鐘)")
            
            creation_flags = 0
            if os.name == 'nt':
                creation_flags = subprocess.CREATE_NO_WINDOW
            
            self.demucs_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                creationflags=creation_flags
            )
            
            output_thread = threading.Thread(
                target=self._read_demucs_output,
                args=(self.demucs_process,),
                daemon=True
            )
            output_thread.start()
            
            self.demucs_process.wait()
            output_thread.join(timeout=2)
            
            if self.demucs_process.returncode == 0:
                self.progress_val.set(100)
                self.log("✓ 分離完成!")
                
                temp_basename = os.path.splitext(temp_name)[0]
                track_dir = os.path.join(job_output, model, temp_basename)
                
                self.log(f"→ 載入音軌: {track_dir}")
                
                if not os.path.exists(track_dir):
                    alt_paths = [
                        os.path.join(job_output, temp_basename),
                        os.path.join(job_output, model, os.path.basename(input_path).replace('.wav', ''))
                    ]
                    for alt in alt_paths:
                        if os.path.exists(alt):
                            track_dir = alt
                            break
                
                all_found = True
                for name in self.tracks.keys():
                    track_path = os.path.join(track_dir, f"{name}.wav")
                    if os.path.exists(track_path):
                        size = os.path.getsize(track_path) / (1024*1024)
                        self.tracks[name]["file"] = track_path
                        self.log(f"  ✓ {name}.wav ({size:.1f} MB)")
                    else:
                        all_found = False
                        self.log(f"  ✗ {name}.wav 不存在")
                
                if all_found:
                    self.log("→ 載入音訊資料到記憶體...")
                    self._load_audio_segments()
                    self.log("✓ 完成! 可以播放並跳轉了")
                    self.lbl_status.config(text="完成")
                    messagebox.showinfo("成功", "分離完成!")
                else:
                    self.log("✗ 部分檔案遺失")
                    self.lbl_status.config(text="不完整")
                    
            elif self.demucs_process.returncode == -15:
                self.log("! 已取消")
            else:
                self.log(f"✗ 錯誤碼: {self.demucs_process.returncode}")
                self.lbl_status.config(text=f"失敗")
                
        except FileNotFoundError:
            self.log("✗ 找不到 demucs")
            self.log("  請執行: pip install demucs")
            messagebox.showerror("錯誤", "找不到 demucs\n請安裝: pip install demucs")
        except Exception as e:
            self.log(f"✗ 錯誤: {e}")
            messagebox.showerror("錯誤", f"{e}")
            
        finally:
            if os.path.exists(temp_name):
                try:
                    os.remove(temp_name)
                except:
                    pass
            
            self.is_processing = False
            self.btn_start_ai.config(state="normal")
            self.btn_cancel.config(state="disabled")
            self.demucs_process = None

    def _read_demucs_output(self, process):
        try:
            progress = 0
            for line in iter(process.stdout.readline, ''):
                if not line:
                    break
                line = line.strip()
                if line and any(k in line.lower() for k in ['progress', 'error', 'separating', 'done']):
                    self.log(f"  {line}")
                if progress < 95:
                    progress += 0.5
                    self.progress_val.set(progress)
        except:
            pass

    def _load_audio_segments(self):
        """載入完整音訊資料以支援跳轉"""
        self.audio_segments = {}
        try:
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    self.log(f"  → 載入 {name}...")
                    self.audio_segments[name] = AudioSegment(data["file"])
            self.log("  ✓ 音訊資料已載入")
        except Exception as e:
            self.log(f"  ! 載入錯誤: {e}")
            self.log(f"  ! 嘗試安裝: pip install scipy")
            # 降級為直接載入 Sound (不支援跳轉)
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    try:
                        data["sound"] = pygame.mixer.Sound(data["file"])
                    except:
                        pass

    def toggle_play(self):
        if not self.audio_segments and not any(t["sound"] for t in self.tracks.values()):
            messagebox.showwarning("提示", "請先完成分離")
            return
        if self.is_playing:
            self.pause_playback()
        else:
            self.start_playback()

    def start_playback(self):
        """開始播放 - 支援從任意位置開始"""
        # 清理舊的臨時檔案
        self._cleanup_temp_files()
        
        pygame.mixer.stop()
        
        start_pos = self.current_pos
        
        for i, (name, data) in enumerate(self.tracks.items()):
            if not data["var"].get():
                continue
            
            try:
                if name in self.audio_segments:
                    # 使用音訊片段 (支援跳轉)
                    temp_file = self.audio_segments[name].get_segment(start_pos)
                    self.temp_files.append(temp_file)
                    sound = pygame.mixer.Sound(temp_file)
                    chan = pygame.mixer.Channel(i)
                    chan.play(sound)
                elif data["sound"]:
                    # 降級模式 (不支援跳轉)
                    chan = pygame.mixer.Channel(i)
                    chan.play(data["sound"])
                else:
                    continue
                    
            except Exception as e:
                self.log(f"! 播放 {name} 失敗: {e}")
        
        self.is_playing = True
        self.start_tick = time.time()
        self.run_sync_loop()

    def pause_playback(self):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            pygame.mixer.pause()
            self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_ui_display(0)
        self._cleanup_temp_files()

    def on_track_change(self):
        """音軌開關改變"""
        if self.is_playing:
            # 記錄當前位置
            self.current_pos += (time.time() - self.start_tick)
            pygame.mixer.stop()
            self.is_playing = False
            # 從當前位置重新播放
            self.start_playback()

    def seek(self, seconds):
        """跳轉功能"""
        # 計算新位置
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
        
        self.current_pos = max(0, min(self.current_pos + seconds, self.total_duration))
        
        # 重新播放
        if self.is_playing:
            pygame.mixer.stop()
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def run_sync_loop(self):
        if self.is_playing:
            now_pos = self.current_pos + (time.time() - self.start_tick)
            if now_pos >= self.total_duration:
                self.stop_playback()
            else:
                self.update_ui_display(now_pos)
                self.root.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        cur_m, cur_s = divmod(int(pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0:
            self.play_pb['value'] = (pos / self.total_duration) * 100

    def _cleanup_temp_files(self):
        """清理臨時音訊檔案"""
        for temp_file in self.temp_files:
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except:
                pass
        self.temp_files = []

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicFilterTest(root)
    root.mainloop()

C:\Users\jt725\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [7]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, scrolledtext
import os
import subprocess
import threading
import pygame
import wave
import shutil
import time
import uuid
import io

# --- 核心參數設定 ---
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

# 嘗試導入 pydub (更好的音訊處理)
try:
    from pydub import AudioSegment as PyDubSegment
    from pydub.playback import _play_with_simpleaudio
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False
    print("警告: pydub 未安裝,跳轉功能受限。建議執行: pip install pydub")

class MusicFilterTest:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂過濾器 - 同步優化版")
        self.root.geometry("700x720")
        
        # 初始化音訊設定
        pygame.mixer.pre_init(44100, -16, 2, 512)
        pygame.mixer.init()
        pygame.mixer.set_num_channels(8)
        
        self.input_wav = tk.StringVar(value="")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        
        # 播放控制變數
        self.current_pos = 0.0
        self.start_tick = 0
        self.is_playing = False
        self.total_duration = 0.0
        
        # 音軌路徑與控制
        self.tracks = {
            "bass":   {"var": tk.BooleanVar(value=True), "file": "", "audio": None, "channel": None},
            "drums":  {"var": tk.BooleanVar(value=True), "file": "", "audio": None, "channel": None},
            "other":  {"var": tk.BooleanVar(value=True), "file": "", "audio": None, "channel": None},
            "vocals": {"var": tk.BooleanVar(value=True), "file": "", "audio": None, "channel": None}
        }
        
        self.demucs_process = None
        self.use_pydub = PYDUB_AVAILABLE
        
        self.setup_ui()

    def setup_ui(self):
        # 1. 檔案選擇
        frame_file = ttk.LabelFrame(self.root, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        # 2. AI 分離控制
        frame_ai = ttk.LabelFrame(self.root, text=" 2. AI 樂器分離 ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        
        # 模型選擇
        model_frame = ttk.Frame(frame_ai)
        model_frame.pack(fill="x", pady=5)
        ttk.Label(model_frame, text="模型:").pack(side="left", padx=5)
        self.model_var = tk.StringVar(value="htdemucs")
        ttk.Radiobutton(model_frame, text="htdemucs (快)", variable=self.model_var, value="htdemucs").pack(side="left", padx=5)
        ttk.Radiobutton(model_frame, text="mdx_extra (優)", variable=self.model_var, value="mdx_extra").pack(side="left", padx=5)
        
        # 裝置選擇
        device_frame = ttk.Frame(frame_ai)
        device_frame.pack(fill="x", pady=5)
        ttk.Label(device_frame, text="裝置:").pack(side="left", padx=5)
        self.device_var = tk.StringVar(value="cpu")
        ttk.Radiobutton(device_frame, text="CPU", variable=self.device_var, value="cpu").pack(side="left", padx=5)
        ttk.Radiobutton(device_frame, text="CUDA", variable=self.device_var, value="cuda").pack(side="left", padx=5)
        
        btn_container = ttk.Frame(frame_ai)
        btn_container.pack(fill="x", pady=5)
        self.btn_start_ai = ttk.Button(btn_container, text="啟動分離", command=self.start_separation_thread)
        self.btn_start_ai.pack(side="left", padx=5)
        self.btn_cancel = ttk.Button(btn_container, text="取消", command=self.cancel_processing, state="disabled")
        self.btn_cancel.pack(side="left", padx=5)
        
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        # 3. 日誌視窗
        frame_log = ttk.LabelFrame(self.root, text=" 處理日誌 ", padding=10)
        frame_log.pack(fill="both", expand=True, padx=10, pady=5)
        
        self.log_text = scrolledtext.ScrolledText(frame_log, height=6, wrap=tk.WORD, font=("Consolas", 9))
        self.log_text.pack(fill="both", expand=True)
        
        ttk.Button(frame_log, text="清除日誌", command=lambda: self.log_text.delete(1.0, tk.END)).pack(pady=5)

        # 4. 播放控制
        frame_play = ttk.LabelFrame(self.root, text=" 3. 同步播放控制 ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        
        # 狀態提示
        pydub_status = "✓ 支援精確跳轉" if PYDUB_AVAILABLE else "⚠ 建議安裝 pydub 以支援跳轉"
        ttk.Label(frame_play, text=pydub_status, font=("Arial", 9), foreground="green" if PYDUB_AVAILABLE else "orange").pack()
        
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 10s", command=lambda: self.seek(-10)).grid(row=0, column=0, padx=2)
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=1, padx=2)
        ttk.Button(btn_box, text="<< 1s", command=lambda: self.seek(-1)).grid(row=0, column=2, padx=2)
        self.btn_play = ttk.Button(btn_box, text="▶ 播放", command=self.toggle_play, width=10)
        self.btn_play.grid(row=0, column=3, padx=5)
        ttk.Button(btn_box, text=">> 1s", command=lambda: self.seek(1)).grid(row=0, column=4, padx=2)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=5, padx=2)
        ttk.Button(btn_box, text=">> 10s", command=lambda: self.seek(10)).grid(row=0, column=6, padx=2)
        ttk.Button(btn_box, text="■ 停止", command=self.stop_playback).grid(row=0, column=7, padx=2)

        # 5. 音軌開關
        frame_tracks = ttk.LabelFrame(self.root, text=" 4. 樂器選擇 ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        for name, data in self.tracks.items():
            cb = ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                                command=self.on_track_change)
            cb.pack(side="left", expand=True)

    def log(self, message):
        timestamp = time.strftime("%H:%M:%S")
        self.log_text.insert(tk.END, f"[{timestamp}] {message}\n")
        self.log_text.see(tk.END)
        self.root.update_idletasks()

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav"), ("所有檔案", "*.*")])
        if path:
            self.input_wav.set(path)
            try:
                with wave.open(path, 'r') as f:
                    self.total_duration = f.getnframes() / float(f.getframerate())
                self.log(f"✓ 已載入: {os.path.basename(path)} ({int(self.total_duration//60)}:{int(self.total_duration%60):02d})")
                self.stop_playback()
            except Exception as e:
                self.log(f"✗ 讀取失敗: {e}")
                messagebox.showerror("錯誤", f"無法讀取檔案:\n{e}")

    def start_separation_thread(self):
        if not self.input_wav.get():
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: 
            messagebox.showwarning("提示", "正在處理中")
            return
        
        self.stop_playback()
        self.progress_val.set(0)
        self.log("=" * 50)
        self.log(f"開始分離 (模型: {self.model_var.get()}, 裝置: {self.device_var.get()})")
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def cancel_processing(self):
        if self.demucs_process and self.demucs_process.poll() is None:
            self.log("! 正在取消...")
            try:
                self.demucs_process.terminate()
                time.sleep(1)
                if self.demucs_process.poll() is None:
                    self.demucs_process.kill()
                self.log("✓ 已取消")
            except Exception as e:
                self.log(f"! 取消錯誤: {e}")
            
            self.lbl_status.config(text="已取消")
            self.is_processing = False
            self.btn_start_ai.config(state="normal")
            self.btn_cancel.config(state="disabled")

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.btn_cancel.config(state="normal")
        self.lbl_status.config(text="處理中...")
        
        input_path = self.input_wav.get()
        temp_name = f"filter_temp_{uuid.uuid4().hex[:8]}.wav"
        
        try:
            self.log(f"→ 複製檔案...")
            shutil.copy(input_path, temp_name)
            file_size = os.path.getsize(temp_name) / (1024*1024)
            self.log(f"✓ 暫存: {temp_name} ({file_size:.1f} MB)")
            
            job_output = os.path.join(OUTPUT_BASE, "filter_session")
            if os.path.exists(job_output):
                shutil.rmtree(job_output)
            os.makedirs(job_output, exist_ok=True)
            self.log(f"✓ 輸出: {job_output}")

            model = self.model_var.get()
            device = self.device_var.get()
            cmd = ['demucs', '-d', device, '-n', model, temp_name, '-o', job_output]
            
            self.log(f"→ 執行 demucs (請耐心等候...)")
            
            creation_flags = 0
            if os.name == 'nt':
                creation_flags = subprocess.CREATE_NO_WINDOW
            
            self.demucs_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                creationflags=creation_flags
            )
            
            output_thread = threading.Thread(
                target=self._read_demucs_output,
                args=(self.demucs_process,),
                daemon=True
            )
            output_thread.start()
            
            self.demucs_process.wait()
            output_thread.join(timeout=2)
            
            if self.demucs_process.returncode == 0:
                self.progress_val.set(100)
                self.log("✓ 分離完成!")
                
                temp_basename = os.path.splitext(temp_name)[0]
                track_dir = os.path.join(job_output, model, temp_basename)
                
                self.log(f"→ 載入音軌...")
                
                if not os.path.exists(track_dir):
                    alt_paths = [
                        os.path.join(job_output, temp_basename),
                        os.path.join(job_output, model, os.path.basename(input_path).replace('.wav', ''))
                    ]
                    for alt in alt_paths:
                        if os.path.exists(alt):
                            track_dir = alt
                            break
                
                all_found = True
                for name in self.tracks.keys():
                    track_path = os.path.join(track_dir, f"{name}.wav")
                    if os.path.exists(track_path):
                        size = os.path.getsize(track_path) / (1024*1024)
                        self.tracks[name]["file"] = track_path
                        self.log(f"  ✓ {name}.wav ({size:.1f} MB)")
                    else:
                        all_found = False
                        self.log(f"  ✗ {name}.wav 不存在")
                
                if all_found:
                    self.log("→ 預載入音訊...")
                    self._preload_audio()
                    self.log("✓ 完成! 可以播放了")
                    self.lbl_status.config(text="完成")
                    messagebox.showinfo("成功", "分離完成!")
                else:
                    self.log("✗ 部分檔案遺失")
                    self.lbl_status.config(text="不完整")
                    
            elif self.demucs_process.returncode == -15:
                self.log("! 已取消")
            else:
                self.log(f"✗ 錯誤碼: {self.demucs_process.returncode}")
                self.lbl_status.config(text="失敗")
                
        except FileNotFoundError:
            self.log("✗ 找不到 demucs")
            self.log("  請執行: pip install demucs")
            messagebox.showerror("錯誤", "找不到 demucs\n請安裝: pip install demucs")
        except Exception as e:
            self.log(f"✗ 錯誤: {e}")
            messagebox.showerror("錯誤", f"{e}")
            
        finally:
            if os.path.exists(temp_name):
                try:
                    os.remove(temp_name)
                except:
                    pass
            
            self.is_processing = False
            self.btn_start_ai.config(state="normal")
            self.btn_cancel.config(state="disabled")
            self.demucs_process = None

    def _read_demucs_output(self, process):
        try:
            progress = 0
            for line in iter(process.stdout.readline, ''):
                if not line:
                    break
                line = line.strip()
                if line and any(k in line.lower() for k in ['progress', 'error', 'separating', 'done']):
                    self.log(f"  {line}")
                if progress < 95:
                    progress += 0.5
                    self.progress_val.set(progress)
        except:
            pass

    def _preload_audio(self):
        """預載入音訊"""
        if PYDUB_AVAILABLE:
            # 使用 pydub 載入 (支援跳轉)
            self.log("  使用 pydub 載入...")
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    try:
                        data["audio"] = PyDubSegment.from_wav(data["file"])
                        self.log(f"    ✓ {name}")
                    except Exception as e:
                        self.log(f"    ✗ {name}: {e}")
        else:
            # 降級為 pygame.mixer.Sound (不支援精確跳轉)
            self.log("  使用 pygame 載入 (不支援跳轉)...")
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    try:
                        data["audio"] = pygame.mixer.Sound(data["file"])
                        self.log(f"    ✓ {name}")
                    except Exception as e:
                        self.log(f"    ✗ {name}: {e}")

    def toggle_play(self):
        if not any(t["audio"] for t in self.tracks.values()):
            messagebox.showwarning("提示", "請先完成分離")
            return
        
        if self.is_playing:
            self.pause_playback()
            self.btn_play.config(text="▶ 播放")
        else:
            self.start_playback()
            self.btn_play.config(text="⏸ 暫停")

    def start_playback(self):
        """開始播放"""
        pygame.mixer.stop()
        
        start_ms = int(self.current_pos * 1000)
        
        if PYDUB_AVAILABLE and start_ms > 0:
            # 使用 pydub 實現精確跳轉
            for i, (name, data) in enumerate(self.tracks.items()):
                if not data["var"].get() or not data["audio"]:
                    continue
                
                try:
                    # 從指定位置開始的音訊片段
                    segment = data["audio"][start_ms:]
                    
                    # 轉換為 pygame Sound
                    # 使用記憶體緩衝區避免建立檔案
                    audio_data = segment.raw_data
                    sound = pygame.mixer.Sound(buffer=audio_data)
                    
                    chan = pygame.mixer.Channel(i)
                    chan.play(sound)
                    data["channel"] = chan
                    
                except Exception as e:
                    self.log(f"! 播放 {name} 失敗: {e}")
        else:
            # 基本播放模式 (從頭開始或降級模式)
            for i, (name, data) in enumerate(self.tracks.items()):
                if not data["var"].get() or not data["audio"]:
                    continue
                
                try:
                    chan = pygame.mixer.Channel(i)
                    chan.play(data["audio"])
                    data["channel"] = chan
                except Exception as e:
                    self.log(f"! 播放 {name} 失敗: {e}")
        
        self.is_playing = True
        self.start_tick = time.time()
        self.run_sync_loop()

    def pause_playback(self):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            pygame.mixer.pause()
            self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_ui_display(0)
        self.btn_play.config(text="▶ 播放")

    def on_track_change(self):
        """音軌開關改變 - 即時更新混音"""
        if self.is_playing:
            # 記錄當前位置
            elapsed = time.time() - self.start_tick
            self.current_pos += elapsed
            
            # 重新播放
            pygame.mixer.stop()
            self.start_playback()

    def seek(self, seconds):
        """跳轉"""
        # 更新位置
        if self.is_playing:
            elapsed = time.time() - self.start_tick
            self.current_pos += elapsed
        
        self.current_pos += seconds
        self.current_pos = max(0, min(self.current_pos, self.total_duration))
        
        # 如果正在播放,重新開始
        if self.is_playing:
            pygame.mixer.stop()
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def run_sync_loop(self):
        if self.is_playing:
            elapsed = time.time() - self.start_tick
            now_pos = self.current_pos + elapsed
            
            if now_pos >= self.total_duration:
                self.stop_playback()
            else:
                self.update_ui_display(now_pos)
                self.root.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        cur_m, cur_s = divmod(int(pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0:
            self.play_pb['value'] = (pos / self.total_duration) * 100

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicFilterTest(root)
    root.mainloop()
#+++++++++++++++++++++++++++++++++++++++++++

In [2]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import os
import subprocess
import threading
import pygame
import wave
import shutil
import time

# --- 核心參數設定 ---
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

class MusicFilterTest:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂過濾器 - 簡化版")
        self.root.geometry("600x550")
        
        # 初始化音訊設定
        pygame.mixer.pre_init(44100, -16, 2, 512)
        pygame.mixer.init()
        pygame.mixer.set_num_channels(8)
        
        self.input_wav = tk.StringVar(value="")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        
        # 播放控制變數
        self.current_pos = 0.0
        self.start_tick = 0
        self.is_playing = False
        self.total_duration = 0.0
        
        # 音軌路徑與控制
        self.tracks = {
            "bass":   {"var": tk.BooleanVar(value=True), "file": ""},
            "drums":  {"var": tk.BooleanVar(value=True), "file": ""},
            "other":  {"var": tk.BooleanVar(value=True), "file": ""},
            "vocals": {"var": tk.BooleanVar(value=True), "file": ""}
        }
        
        # 儲存預載的音訊
        self.sounds = {}
        
        self.setup_ui()

    def setup_ui(self):
        # 1. 檔案選擇
        frame_file = ttk.LabelFrame(self.root, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        # 2. AI 分離控制
        frame_ai = ttk.LabelFrame(self.root, text=" 2. AI 樂器分離 (自動清理舊暫存) ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        self.btn_start_ai = ttk.Button(frame_ai, text="啟動分離 (Demucs)", command=self.start_separation_thread)
        self.btn_start_ai.pack(pady=5)
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        # 3. 播放控制與進度條
        frame_play = ttk.LabelFrame(self.root, text=" 3. 同步播放控制 ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=5)
        self.btn_play = ttk.Button(btn_box, text="播放 / 暫停", command=self.toggle_play)
        self.btn_play.grid(row=0, column=1, padx=5)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=2, padx=5)

        # 4. 音軌開關
        frame_tracks = ttk.LabelFrame(self.root, text=" 4. 樂器選擇 ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        for name, data in self.tracks.items():
            ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                            command=self.on_track_change).pack(side="left", expand=True)

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav")])
        if path:
            self.input_wav.set(path)
            with wave.open(path, 'r') as f:
                self.total_duration = f.getnframes() / float(f.getframerate())
            self.stop_playback()

    def start_separation_thread(self):
        if not self.input_wav.get():
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: return
        
        self.stop_playback()
        self.progress_val.set(0)
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.lbl_status.config(text="AI 正在處理中，請稍候...")
        
        input_path = self.input_wav.get()
        temp_name = "filter_temp.wav"
        shutil.copy(input_path, temp_name)
        
        # 建立專屬輸出資料夾
        job_output = os.path.join(OUTPUT_BASE, "filter_session")
        if os.path.exists(job_output):
            shutil.rmtree(job_output)

        cmd = ['demucs', '-d', 'cpu', '-n', 'htdemucs', temp_name, '-o', job_output]
        
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            
            # 簡單進度條
            while process.poll() is None:
                curr = self.progress_val.get()
                if curr < 90: self.progress_val.set(curr + 0.5)
                self.root.update_idletasks()
                time.sleep(0.1)
            
            if process.returncode == 0:
                self.progress_val.set(100)
                # 設定音軌路徑
                track_dir = os.path.join(job_output, "htdemucs", "filter_temp")
                for name in self.tracks.keys():
                    self.tracks[name]["file"] = os.path.join(track_dir, f"{name}.wav")
                
                # 預載音訊
                self.sounds = {}
                for name, data in self.tracks.items():
                    if os.path.exists(data["file"]):
                        self.sounds[name] = pygame.mixer.Sound(data["file"])
                
                self.lbl_status.config(text="分離完成！可以開始播放。")
                messagebox.showinfo("成功", "樂器分離已完成")
            else:
                self.lbl_status.config(text="分離失敗。")
        except Exception as e:
            self.lbl_status.config(text=f"錯誤: {e}")
        finally:
            if os.path.exists(temp_name): os.remove(temp_name)
            self.is_processing = False
            self.btn_start_ai.config(state="normal")

    def toggle_play(self):
        if not self.sounds:
            messagebox.showwarning("提示", "請先點擊啟動分離功能")
            return
        if self.is_playing:
            self.pause_playback()
        else:
            self.start_playback()

    def start_playback(self):
        """開始播放 - 修正版"""
        pygame.mixer.stop()
        
        # 同時啟動所有選中的音軌
        for i, (name, data) in enumerate(self.tracks.items()):
            if data["var"].get() and name in self.sounds:
                chan = pygame.mixer.Channel(i)
                chan.play(self.sounds[name])
        
        self.is_playing = True
        self.start_tick = time.time() - self.current_pos  # 修正: 考慮已播放的時間
        self.run_sync_loop()

    def pause_playback(self):
        if self.is_playing:
            self.current_pos = time.time() - self.start_tick  # 計算實際播放位置
            pygame.mixer.pause()
            self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_ui_display(0)

    def on_track_change(self):
        """音軌開關改變"""
        if self.is_playing:
            # 保持當前位置
            self.current_pos = time.time() - self.start_tick
            pygame.mixer.stop()
            # 立即重新播放
            for i, (name, data) in enumerate(self.tracks.items()):
                if data["var"].get() and name in self.sounds:
                    chan = pygame.mixer.Channel(i)
                    chan.play(self.sounds[name])

    def seek(self, seconds):
        """跳轉 - 修正版"""
        was_playing = self.is_playing
        
        # 更新位置
        if self.is_playing:
            self.current_pos = time.time() - self.start_tick
        
        self.current_pos += seconds
        
        # 限制範圍
        if self.current_pos < 0: 
            self.current_pos = 0
        if self.current_pos > self.total_duration: 
            self.current_pos = self.total_duration
        
        # 重新播放
        if was_playing:
            pygame.mixer.stop()
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def run_sync_loop(self):
        """同步循環"""
        if self.is_playing:
            now_pos = time.time() - self.start_tick
            if now_pos >= self.total_duration:
                self.stop_playback()
            else:
                self.update_ui_display(now_pos)
                self.root.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        """更新顯示"""
        cur_m, cur_s = divmod(int(pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0:
            self.play_pb['value'] = (pos / self.total_duration) * 100

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicFilterTest(root)
    root.mainloop()

In [5]:
#完成
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import os
import subprocess
import threading
import pygame
import wave
import shutil
import time
import uuid

# --- 核心參數設定 ---
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

# 嘗試導入 pydub (支援跳轉)
try:
    from pydub import AudioSegment as PyDubSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False

class MusicFilterTest:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂過濾器")
        self.root.geometry("600x500")
        
        # 初始化音訊設定
        pygame.mixer.pre_init(44100, -16, 2, 512)
        pygame.mixer.init()
        pygame.mixer.set_num_channels(8)
        
        self.input_wav = tk.StringVar(value="")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        
        # 播放控制變數
        self.current_pos = 0.0
        self.start_tick = 0
        self.is_playing = False
        self.total_duration = 0.0
        
        # 音軌
        self.tracks = {
            "bass":   {"var": tk.BooleanVar(value=True), "file": "", "audio": None},
            "drums":  {"var": tk.BooleanVar(value=True), "file": "", "audio": None},
            "other":  {"var": tk.BooleanVar(value=True), "file": "", "audio": None},
            "vocals": {"var": tk.BooleanVar(value=True), "file": "", "audio": None}
        }
        
        self.demucs_process = None
        
        self.setup_ui()

    def setup_ui(self):
        # 1. 檔案選擇
        frame_file = ttk.LabelFrame(self.root, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        # 2. AI 分離控制
        frame_ai = ttk.LabelFrame(self.root, text=" 2. AI 樂器分離 ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        self.btn_start_ai = ttk.Button(frame_ai, text="啟動分離 (Demucs)", command=self.start_separation_thread)
        self.btn_start_ai.pack(pady=5)
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        # 3. 播放控制
        frame_play = ttk.LabelFrame(self.root, text=" 3. 同步播放控制 ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=5)
        self.btn_play = ttk.Button(btn_box, text="播放 / 暫停", command=self.toggle_play)
        self.btn_play.grid(row=0, column=1, padx=5)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=2, padx=5)

        # 4. 音軌選擇
        frame_tracks = ttk.LabelFrame(self.root, text=" 4. 樂器選擇 (即時切換) ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        for name, data in self.tracks.items():
            ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                            command=self.on_track_change).pack(side="left", expand=True)

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav")])
        if path:
            self.input_wav.set(path)
            try:
                with wave.open(path, 'r') as f:
                    self.total_duration = f.getnframes() / float(f.getframerate())
                self.stop_playback()
            except Exception as e:
                messagebox.showerror("錯誤", f"無法讀取檔案:\n{e}")

    def start_separation_thread(self):
        if not self.input_wav.get():
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: 
            return
        
        self.stop_playback()
        self.progress_val.set(0)
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.lbl_status.config(text="AI 正在處理中，請稍候...")
        
        input_path = self.input_wav.get()
        temp_name = f"filter_temp_{uuid.uuid4().hex[:8]}.wav"
        
        try:
            shutil.copy(input_path, temp_name)
            
            job_output = os.path.join(OUTPUT_BASE, "filter_session")
            if os.path.exists(job_output):
                shutil.rmtree(job_output)
            os.makedirs(job_output, exist_ok=True)

            cmd = ['demucs', '-d', 'cpu', '-n', 'htdemucs', temp_name, '-o', job_output]
            
            creation_flags = 0
            if os.name == 'nt':
                creation_flags = subprocess.CREATE_NO_WINDOW
            
            self.demucs_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                creationflags=creation_flags
            )
            
            # 背景執行緒讀取輸出避免卡住
            def read_output():
                for line in iter(self.demucs_process.stdout.readline, ''):
                    if not line:
                        break
            
            output_thread = threading.Thread(target=read_output, daemon=True)
            output_thread.start()
            
            # 進度條
            while self.demucs_process.poll() is None:
                curr = self.progress_val.get()
                if curr < 90: 
                    self.progress_val.set(curr + 0.5)
                self.root.update_idletasks()
                time.sleep(0.1)
            
            if self.demucs_process.returncode == 0:
                self.progress_val.set(100)
                
                temp_basename = os.path.splitext(temp_name)[0]
                track_dir = os.path.join(job_output, "htdemucs", temp_basename)
                
                # 載入音軌
                all_found = True
                for name in self.tracks.keys():
                    track_path = os.path.join(track_dir, f"{name}.wav")
                    if os.path.exists(track_path):
                        self.tracks[name]["file"] = track_path
                    else:
                        all_found = False
                
                if all_found:
                    self._preload_audio()
                    self.lbl_status.config(text="分離完成！可以開始播放。")
                    messagebox.showinfo("成功", "樂器分離已完成")
                else:
                    self.lbl_status.config(text="分離完成但部分檔案遺失")
                    
            else:
                self.lbl_status.config(text="分離失敗")
                
        except FileNotFoundError:
            self.lbl_status.config(text="找不到 demucs")
            messagebox.showerror("錯誤", "找不到 demucs\n請執行: pip install demucs")
        except Exception as e:
            self.lbl_status.config(text=f"錯誤: {e}")
            messagebox.showerror("錯誤", str(e))
            
        finally:
            if os.path.exists(temp_name):
                try:
                    os.remove(temp_name)
                except:
                    pass
            
            self.is_processing = False
            self.btn_start_ai.config(state="normal")
            self.demucs_process = None

    def _preload_audio(self):
        """預載入音訊"""
        if PYDUB_AVAILABLE:
            # 使用 pydub (支援跳轉)
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    try:
                        data["audio"] = PyDubSegment.from_wav(data["file"])
                    except:
                        pass
        else:
            # 使用 pygame (基本播放)
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    try:
                        data["audio"] = pygame.mixer.Sound(data["file"])
                    except:
                        pass

    def toggle_play(self):
        if not any(t["audio"] for t in self.tracks.values()):
            messagebox.showwarning("提示", "請先完成分離")
            return
        
        if self.is_playing:
            self.pause_playback()
        else:
            self.start_playback()

    def start_playback(self):
        """開始播放 - 支援跳轉和同步"""
        pygame.mixer.stop()
        
        start_ms = int(self.current_pos * 1000)
        
        if PYDUB_AVAILABLE and start_ms > 0:
            # pydub 模式 - 支援精確跳轉
            for i, (name, data) in enumerate(self.tracks.items()):
                if not data["var"].get() or not data["audio"]:
                    continue
                
                try:
                    # 從指定位置開始
                    segment = data["audio"][start_ms:]
                    sound = pygame.mixer.Sound(buffer=segment.raw_data)
                    chan = pygame.mixer.Channel(i)
                    chan.play(sound)
                except:
                    pass
        else:
            # pygame 基本模式
            for i, (name, data) in enumerate(self.tracks.items()):
                if not data["var"].get() or not data["audio"]:
                    continue
                
                try:
                    chan = pygame.mixer.Channel(i)
                    chan.play(data["audio"])
                except:
                    pass
        
        self.is_playing = True
        self.start_tick = time.time()
        self.run_sync_loop()

    def pause_playback(self):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            pygame.mixer.pause()
            self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_ui_display(0)

    def on_track_change(self):
        """音軌開關改變 - 即時更新混音"""
        if self.is_playing:
            # 記錄當前位置
            self.current_pos += (time.time() - self.start_tick)
            # 重新播放
            pygame.mixer.stop()
            self.start_playback()

    def seek(self, seconds):
        """跳轉功能"""
        # 更新位置
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
        
        self.current_pos += seconds
        self.current_pos = max(0, min(self.current_pos, self.total_duration))
        
        # 重新播放
        if self.is_playing:
            pygame.mixer.stop()
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def run_sync_loop(self):
        """同步循環"""
        if self.is_playing:
            elapsed = time.time() - self.start_tick
            now_pos = self.current_pos + elapsed
            
            if now_pos >= self.total_duration:
                self.stop_playback()
            else:
                self.update_ui_display(now_pos)
                self.root.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        """更新顯示"""
        cur_m, cur_s = divmod(int(pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0:
            self.play_pb['value'] = (pos / self.total_duration) * 100

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicFilterTest(root)
    root.mainloop()

In [1]:
#完全體!!!!!!!!!!!!!!!!!!!!!!!!
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import os
import subprocess
import threading
import pygame
import wave
import shutil
import time
import uuid

# --- 核心參數設定 ---
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

# 嘗試導入 pydub (支援跳轉)
try:
    from pydub import AudioSegment as PyDubSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False

class MusicFilterTest:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂過濾器")
        self.root.geometry("600x520")
        
        # 初始化音訊設定
        pygame.mixer.pre_init(44100, -16, 2, 512)
        pygame.mixer.init()
        pygame.mixer.set_num_channels(8)
        
        self.input_wav = tk.StringVar(value="")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        
        # 播放控制變數
        self.current_pos = 0.0
        self.start_tick = 0
        self.is_playing = False
        self.total_duration = 0.0
        
        # 音軌
        self.tracks = {
            "bass":   {"var": tk.BooleanVar(value=True), "file": "", "audio": None},
            "drums":  {"var": tk.BooleanVar(value=True), "file": "", "audio": None},
            "other":  {"var": tk.BooleanVar(value=True), "file": "", "audio": None},
            "vocals": {"var": tk.BooleanVar(value=True), "file": "", "audio": None}
        }
        
        self.demucs_process = None
        self.setup_ui()

    def setup_ui(self):
        # 1. 檔案選擇
        frame_file = ttk.LabelFrame(self.root, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        # 2. AI 分離控制
        frame_ai = ttk.LabelFrame(self.root, text=" 2. AI 樂器分離 ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        
        # 顯示加速選項
        self.device_var = tk.StringVar(value="cuda") # 預設改為 CUDA
        device_box = ttk.Frame(frame_ai)
        device_box.pack(pady=2)
        ttk.Radiobutton(device_box, text="GPU (CUDA 加速)", variable=self.device_var, value="cuda").pack(side="left", padx=10)
        ttk.Radiobutton(device_box, text="CPU (較慢)", variable=self.device_var, value="cpu").pack(side="left", padx=10)

        self.btn_start_ai = ttk.Button(frame_ai, text="啟動分離 (Demucs)", command=self.start_separation_thread)
        self.btn_start_ai.pack(pady=5)
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        # 3. 播放控制
        frame_play = ttk.LabelFrame(self.root, text=" 3. 同步播放控制 ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=5)
        self.btn_play = ttk.Button(btn_box, text="播放 / 暫停", command=self.toggle_play)
        self.btn_play.grid(row=0, column=1, padx=5)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=2, padx=5)

        # 4. 音軌選擇
        frame_tracks = ttk.LabelFrame(self.root, text=" 4. 樂器選擇 (即時切換) ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        for name, data in self.tracks.items():
            ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                            command=self.on_track_change).pack(side="left", expand=True)

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav")])
        if path:
            self.input_wav.set(path)
            try:
                with wave.open(path, 'r') as f:
                    self.total_duration = f.getnframes() / float(f.getframerate())
                self.stop_playback()
            except Exception as e:
                messagebox.showerror("錯誤", f"無法讀取檔案:\n{e}")

    def start_separation_thread(self):
        if not self.input_wav.get():
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: 
            return
        self.stop_playback()
        self.progress_val.set(0)
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.lbl_status.config(text=f"AI 使用 {self.device_var.get().upper()} 處理中...")
        
        input_path = self.input_wav.get()
        temp_name = f"filter_temp_{uuid.uuid4().hex[:8]}.wav"
        
        try:
            shutil.copy(input_path, temp_name)
            job_output = os.path.join(OUTPUT_BASE, "filter_session")
            if os.path.exists(job_output):
                shutil.rmtree(job_output)
            os.makedirs(job_output, exist_ok=True)

            # 修改點：這裡支援選擇 cuda 或 cpu
            cmd = ['demucs', '-d', self.device_var.get(), '-n', 'htdemucs', temp_name, '-o', job_output]
            
            creation_flags = 0
            if os.name == 'nt':
                creation_flags = subprocess.CREATE_NO_WINDOW
            
            self.demucs_process = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1, creationflags=creation_flags
            )
            
            # 讀取輸出以防止阻塞
            while self.demucs_process.poll() is None:
                self.demucs_process.stdout.readline() # 雖然不顯示，但要讀取
                curr = self.progress_val.get()
                if curr < 95: 
                    self.progress_val.set(curr + 0.5)
                self.root.update_idletasks()
                time.sleep(0.1)
            
            if self.demucs_process.returncode == 0:
                self.progress_val.set(100)
                temp_basename = os.path.splitext(temp_name)[0]
                track_dir = os.path.join(job_output, "htdemucs", temp_basename)
                
                # 載入音軌路徑
                for name in self.tracks.keys():
                    track_path = os.path.join(track_dir, f"{name}.wav")
                    if os.path.exists(track_path):
                        self.tracks[name]["file"] = track_path
                
                self._preload_audio()
                self.lbl_status.config(text="分離完成！可以開始播放。")
                messagebox.showinfo("成功", "樂器分離已完成")
            else:
                self.lbl_status.config(text="分離失敗 (如果是CUDA錯誤請改用CPU嘗試)")
                
        except Exception as e:
            self.lbl_status.config(text=f"錯誤: {e}")
        finally:
            if os.path.exists(temp_name):
                try: os.remove(temp_name)
                except: pass
            self.is_processing = False
            self.btn_start_ai.config(state="normal")

    def _preload_audio(self):
        """預載入音訊，優化：不論有沒有 PyDub 都建立 Sound 物件"""
        for name, data in self.tracks.items():
            if os.path.exists(data["file"]):
                if PYDUB_AVAILABLE:
                    data["audio_segment"] = PyDubSegment.from_wav(data["file"])
                # 預先載入一個 Sound 物件供 0 秒播放
                data["audio_sound"] = pygame.mixer.Sound(data["file"])

    def toggle_play(self):
        if not any(t.get("audio_sound") for t in self.tracks.values()):
            messagebox.showwarning("提示", "請先完成分離")
            return
        if self.is_playing:
            self.pause_playback()
        else:
            self.start_playback()

    def start_playback(self):
        """核心修正：解決初始播放無聲的問題"""
        pygame.mixer.stop()
        
        # 強制等待一小段時間讓 Mixer 穩定 (50ms)
        time.sleep(0.05)
        
        start_ms = int(self.current_pos * 1000)
        
        for i, (name, data) in enumerate(self.tracks.items()):
            if not data["var"].get():
                continue
            
            chan = pygame.mixer.Channel(i)
            
            # 如果有跳轉 (start_ms > 0) 且有 PyDub，使用 Buffer 播放
            if PYDUB_AVAILABLE and start_ms > 0:
                try:
                    segment = data["audio_segment"][start_ms:]
                    sound = pygame.mixer.Sound(buffer=segment.raw_data)
                    chan.play(sound)
                except:
                    pass
            # 如果是從 0 秒開始或沒有 PyDub，直接播放預載入的 Sound
            elif data.get("audio_sound"):
                chan.play(data["audio_sound"])
        
        self.is_playing = True
        self.start_tick = time.time()
        self.run_sync_loop()

    def pause_playback(self):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            pygame.mixer.stop() # 改用 stop 比 pause 在多聲道更同步
            self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_ui_display(0)

    def on_track_change(self):
        """音軌即時同步，保持在當前位置"""
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def seek(self, seconds):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
        
        self.current_pos += seconds
        self.current_pos = max(0, min(self.current_pos, self.total_duration))
        
        if self.is_playing:
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def run_sync_loop(self):
        if self.is_playing:
            elapsed = time.time() - self.start_tick
            now_pos = self.current_pos + elapsed
            
            if now_pos >= self.total_duration:
                self.stop_playback()
            else:
                self.update_ui_display(now_pos)
                self.root.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        cur_m, cur_s = divmod(int(pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0:
            self.play_pb['value'] = (pos / self.total_duration) * 100

if __name__ == "__main__":
    root = tk.Tk()
    app = MusicFilterTest(root)
    root.mainloop()

C:\Users\jt725\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
